# NB12 Anatomy of a real block — Chapter 12

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB12-1: cut the 80-byte genesis block header into six fields (bundled snapshot file; if it is missing, a constant with the same bytes)
import struct, hashlib, datetime, os

HEADER_HEX = ("0100000000000000000000000000000000000000000000000000000000000000000000003ba3edfd7a7b12b27ac72c3e67768f617fc81bc3888a51323a9fb8aa4b1e5e4a29ab5f49ffff001d1dac2b7c")   # the same bytes as data/genesis_header.hex

def load_hex(path, fallback):
    try:
        with open(path) as f: return f.read().strip()
    except OSError:
        return fallback

header = bytes.fromhex(load_hex("data/genesis_header.hex", HEADER_HEX))
print("header length:", len(header), "bytes")
assert header.hex() == HEADER_HEX, "the bundled file and the constant differ"

fields = [("version (nVersion)", 0, 4), ("previous block hash (hashPrevBlock)", 4, 32), ("Merkle root (hashMerkleRoot)", 36, 32),
          ("time (nTime)", 68, 4), ("target (nBits)", 72, 4), ("nonce (nNonce)", 76, 4)]
for name, off, size in fields:
    raw = header[off:off + size]
    print(f"[{off:>2}:{off+size:>2}] {name:<36} {raw.hex()}")

nVersion = struct.unpack("<I", header[0:4])[0]          # "<" = little-endian
hashPrev = header[4:36][::-1].hex()                     # hashes are displayed reversed
merkle   = header[36:68][::-1].hex()
nTime    = struct.unpack("<I", header[68:72])[0]
nBits    = struct.unpack("<I", header[72:76])[0]
nNonce   = struct.unpack("<I", header[76:80])[0]
print()
print("nVersion =", nVersion)
print("hashPrevBlock =", hashPrev, "(0 because this is the genesis block: there is no parent to point to)")
print("hashMerkleRoot =", merkle)
print("nTime =", nTime, "=", datetime.datetime.fromtimestamp(nTime, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"))
print(f"nBits = 0x{nBits:08x}  ({nBits})")
print("nNonce =", nNonce, f"(0x{nNonce:08x})")
print("matches CreateGenesisBlock(1231006505, 2083236893, 0x1d00ffff, 1, 50*COIN) in the Core source:",
      (nTime, nNonce, nBits, nVersion) == (1231006505, 2083236893, 0x1d00ffff, 1))

header length: 80 bytes
[ 0: 4] version (nVersion)                   01000000
[ 4:36] previous block hash (hashPrevBlock)  0000000000000000000000000000000000000000000000000000000000000000
[36:68] Merkle root (hashMerkleRoot)         3ba3edfd7a7b12b27ac72c3e67768f617fc81bc3888a51323a9fb8aa4b1e5e4a
[68:72] time (nTime)                         29ab5f49
[72:76] target (nBits)                       ffff001d
[76:80] nonce (nNonce)                       1dac2b7c

nVersion = 1
hashPrevBlock = 0000000000000000000000000000000000000000000000000000000000000000 (0 because this is the genesis block: there is no parent to point to)
hashMerkleRoot = 4a5e1e4baab89f3a32518a88c31bc87f618f76673e2cc77ab2127b7afdeda33b
nTime = 1231006505 = 2009-01-03 18:15:05 UTC
nBits = 0x1d00ffff  (486604799)
nNonce = 2083236893 (0x7c2bac1d)
matches CreateGenesisBlock(1231006505, 2083236893, 0x1d00ffff, 1, 50*COIN) in the Core source: True


In [ ]:
# NB12-2: double SHA-256 → block hash. Does it match the published value, is it below the target, and what happens if the nonce changes?
def sha256d(b: bytes) -> bytes:
    return hashlib.sha256(hashlib.sha256(b).digest()).digest()

KNOWN = "000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f"   # the assert value in Core's chainparams.cpp
block_hash = sha256d(header)[::-1].hex()
print("computed block hash: ", block_hash)
print("published block hash:", KNOWN)
print("match:", block_hash == KNOWN)

def nbits_to_target(nbits: int) -> int:
    return (nbits & 0x007FFFFF) * 256 ** ((nbits >> 24) - 3)
target = nbits_to_target(nBits)
h_int = int(block_hash, 16)
print("target T = 0x%064x" % target)
print("hash ≤ target (valid proof of work):", h_int <= target)
print("number of leading 0 bits in the hash:", 256 - h_int.bit_length(), "(the minimum the target requires is 32)")

# change the nonce by just 1
tampered = header[:76] + struct.pack("<I", nNonce + 1)
t_hash = sha256d(tampered)[::-1].hex()
print("hash of the header with the nonce changed to 2083236894:", t_hash, "→ below the target?", int(t_hash, 16) <= target)

computed block hash:  000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f
published block hash: 000000000019d6689c085ae165831e934ff763ae46a2a6c172b3f1b60a8ce26f
match: True
target T = 0x00000000ffff0000000000000000000000000000000000000000000000000000
hash ≤ target (valid proof of work): True
number of leading 0 bits in the hash: 43 (the minimum the target requires is 32)
hash of the header with the nonce changed to 2083236894: 9b227a4a5daa0cbae6874144bc5d7797d0513e320aceadeb3b06304971a41b1c → below the target? False


In [ ]:
# NB12-3: the whole 285-byte block — transaction count, the inputs and outputs of the coinbase transaction, the sentence inside it, txid == Merkle root
BLOCK_HEX = HEADER_HEX + ("0101000000010000000000000000000000000000000000000000000000000000000000000000ffffffff4d04ffff001d0104455468652054696d65732030332f4a616e2f32303039204368616e63656c6c6f72206f6e206272696e6b206f66207365636f6e64206261696c6f757420666f722062616e6b73ffffffff0100f2052a01000000434104678afdb0fe5548271967f1a67130b7105cd6a828e03909a67962e0ea1f61deb649f6bc3f4cef38c4f35504e51ec112de5c384df7ba0b8d578a4c702b6bf11d5fac00000000")
block = bytes.fromhex(load_hex("data/genesis_block.hex", BLOCK_HEX))
assert block.hex() == BLOCK_HEX and block[:80] == header
print("block length:", len(block), "bytes | header 80 + rest", len(block) - 80)

pos = 80
tx_count = block[pos]; pos += 1                    # transaction count (a variable-length integer; 1 byte here)
print("transaction count:", tx_count)
tx_start = pos
version = struct.unpack("<I", block[pos:pos+4])[0]; pos += 4
in_count = block[pos]; pos += 1
prev_txid = block[pos:pos+32][::-1].hex(); pos += 32
prev_index = struct.unpack("<I", block[pos:pos+4])[0]; pos += 4
script_len = block[pos]; pos += 1
script_sig = block[pos:pos+script_len]; pos += script_len
sequence = struct.unpack("<I", block[pos:pos+4])[0]; pos += 4
print(f"transaction version {version}, input count {in_count}, previous output = ({prev_txid[:8]}…, 0x{prev_index:08x}) ← points to no output = coinbase")
print("input script (scriptSig) length:", script_len, "bytes")

# read the scriptSig in "pushes": if the leading byte is n (1 to 75), put the next n bytes on the stack
i, pushes = 0, []
while i < len(script_sig):
    n = script_sig[i]; pushes.append(script_sig[i+1:i+1+n]); i += 1 + n
for k, p in enumerate(pushes):
    print(f"  push {k}: length {len(p)}  {p.hex() if len(p) <= 4 else p.hex()[:16] + '…'}")
print("  push 0 as a little-endian integer:", int.from_bytes(pushes[0], "little"), "(= the same number as nBits 486604799)")
print("  push 2 as ASCII:", repr(pushes[2].decode("ascii")))

out_count = block[pos]; pos += 1
value = struct.unpack("<Q", block[pos:pos+8])[0]; pos += 8
spk_len = block[pos]; pos += 1
spk = block[pos:pos+spk_len]; pos += spk_len
locktime = struct.unpack("<I", block[pos:pos+4])[0]; pos += 4
print(f"output count {out_count}, amount {value:,} satoshis = {value/1e8} BTC, locking script {spk_len} bytes, locktime {locktime}")
print("  locking script = [push 65 bytes] public key (first byte 0x%02x) … [0x%02x = OP_CHECKSIG]" % (spk[1], spk[-1]))
print("  public key, 65 bytes =", spk[1:66].hex())

tx_bytes = block[tx_start:pos]
print("transaction length:", len(tx_bytes), "bytes | read to the end of the block:", pos == len(block))
txid = sha256d(tx_bytes)[::-1].hex()
print("txid = double SHA-256(transaction) =", txid)
print("Merkle root in the header          =", merkle)
print("with only one transaction, Merkle root == txid :", txid == merkle)

block length: 285 bytes | header 80 + rest 205
transaction count: 1
transaction version 1, input count 1, previous output = (00000000…, 0xffffffff) ← points to no output = coinbase
input script (scriptSig) length: 77 bytes
  push 0: length 4  ffff001d
  push 1: length 1  04
  push 2: length 69  5468652054696d65…
  push 0 as a little-endian integer: 486604799 (= the same number as nBits 486604799)
  push 2 as ASCII: 'The Times 03/Jan/2009 Chancellor on brink of second bailout for banks'
output count 1, amount 5,000,000,000 satoshis = 50.0 BTC, locking script 67 bytes, locktime 0
  locking script = [push 65 bytes] public key (first byte 0x04) … [0xac = OP_CHECKSIG]
  public key, 65 bytes = 04678afdb0fe5548271967f1a67130b7105cd6a828e03909a67962e0ea1f61deb649f6bc3f4cef38c4f35504e51ec112de5c384df7ba0b8d578a4c702b6bf11d5f
transaction length: 204 bytes | read to the end of the block: True
txid = double SHA-256(transaction) = 4a5e1e4baab89f3a32518a88c31bc87f618f76673e2cc77ab2127b7afdeda33b
Mer